In [0]:
%sql
-- Dimensions validation
SELECT 
    'dim_product' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(DISTINCT product_id) AS duplicate_pk_count,
    SUM(CASE
        WHEN product_id IS NULL THEN 1 
        ELSE 0 
    END) AS null_pk_count
FROM instacart.instacart_mart.dim_product

UNION ALL

SELECT 
    'dim_order',
    COUNT(*),
    COUNT(*) - COUNT(DISTINCT order_id),
    SUM(CASE 
        WHEN order_id IS NULL THEN 1 
        ELSE 0 
    END)
FROM instacart.instacart_mart.dim_order

UNION ALL

SELECT 
    'dim_aisle',
    COUNT(*),
    COUNT(*) - COUNT(DISTINCT aisle_id),
    SUM(CASE 
        WHEN aisle_id IS NULL THEN 1 
        ELSE 0 
    END)
FROM instacart.instacart_mart.dim_aisle;

In [0]:
%sql
-- Fact table validation
SELECT 
    'fact_order_products' AS table_name,
    COUNT(*) AS total_rows,

    -- Composite Primary Key check (order_id + product_id must be unique)
    COUNT(*) - COUNT(DISTINCT order_id, product_id) AS duplicate_composite_pk_count,
    
    -- Foreign Key Null Checks
    SUM(CASE 
        WHEN order_id IS NULL THEN 1 
        ELSE 0 
    END) AS null_order_id_count,
    SUM(CASE 
        WHEN product_id IS NULL THEN 1 
        ELSE 0 
    END) AS null_product_id_count,
    SUM(CASE 
        WHEN department_id IS NULL THEN 1 
        ELSE 0 
    END) AS null_department_id_count,
    SUM(CASE 
        WHEN aisle_id IS NULL THEN 1 
        ELSE 0 
    END) AS null_aisle_id_count,
    
    -- Reordered flag should only be 0 or 1
    SUM(CASE 
        WHEN reordered NOT IN (0, 1) OR reordered IS NULL THEN 1 
        ELSE 0 
    END) AS invalid_reordered_flag_count
FROM instacart.instacart_mart.fact_order_products;

In [0]:
%sql
WITH fact_check AS (
    SELECT 
        COUNT(*) - COUNT(DISTINCT order_id, product_id) AS dup_pk,
        SUM(CASE
                WHEN reordered NOT IN (0, 1) OR reordered IS NULL THEN 1 
                ELSE 0 
            END) AS bad_flags
    FROM instacart.instacart_mart.fact_order_products
),
ref_check AS (
    SELECT 
        -- Orphan orders check
        SUM(CASE WHEN o.order_id IS NULL THEN 1 ELSE 0 END) +
        
        -- Orphan products check
        SUM(CASE WHEN p.product_id IS NULL THEN 1 ELSE 0 END) +
        
        -- Orphan aisles check
        SUM(CASE WHEN a.aisle_id IS NULL THEN 1 ELSE 0 END) AS total_orphans
    FROM instacart.instacart_mart.fact_order_products f
    LEFT JOIN instacart.instacart_mart.dim_order o 
        ON f.order_id = o.order_id
    LEFT JOIN instacart.instacart_mart.dim_product p 
        ON f.product_id = p.product_id
    LEFT JOIN instacart.instacart_mart.dim_aisle a 
        ON f.aisle_id = a.aisle_id
)
SELECT 
    'Mart Layer Quality Check' AS suite_name,
    CASE 
        WHEN fact_check.dup_pk = 0 
         AND fact_check.bad_flags = 0 
         AND ref_check.total_orphans = 0 THEN 'PASSED'
        ELSE 'FAILED'
    END AS status,
    fact_check.dup_pk AS duplicate_fact_rows,
    fact_check.bad_flags AS invalid_domain_values,
    ref_check.total_orphans AS orphan_records
FROM fact_check, ref_check;